# Day 03：池化 —— 信息的浓缩与抽象

> 👁️ 第三周 · 视觉的征服 · 第 3 天

前两天我们学会了用卷积提取图像特征。但卷积层产生了一个新问题：**特征图太多了，而且相邻像素的信息高度重复**。

一个 28×28 的图像经过 6 个 3×3 卷积核后，产生 6 张 26×26 的特征图——总共 4,056 个数值。如果再接一个卷积层，计算量会迅速膨胀。

今天，我们要学习**池化（Pooling）**——一种简单但极其有效的降维技术。它不增加任何可学习参数，却能让网络变得更高效、更鲁棒。

**今天的任务**：理解最大池化和平均池化的原理，亲眼看到它们如何压缩信息、提供平移不变性。

---
## 1. 历史背景：特征图太多了怎么办？

### 卷积的「副作用」

卷积层用权值共享解决了参数爆炸，但它带来了一个新问题：**特征图的尺寸仍然很大**。

以 LeNet-5 为例：
- 输入：32×32 的灰度图
- 第一层卷积（6 个 5×5 核）：输出 6 张 28×28 的特征图 = **4,704 个值**
- 第二层卷积（16 个 5×5 核）：如果直接接在第一层后面，输出 16 张 24×24 = **9,216 个值**

如果不做任何处理，特征图的数量和尺寸会迅速膨胀。而且，相邻像素的特征通常非常相似——左上角的「边缘强度」和它右边一个像素的「边缘强度」几乎一样。这些冗余信息不仅浪费计算，还可能导致过拟合。

### 池化的诞生

池化（Pooling）的思想来自 Hubel 和 Wiesel 发现的「复杂细胞」——它们对边缘的方向敏感，但对边缘的**精确位置**不那么敏感。

翻译成算法语言就是：**把特征图中相邻区域的信息「汇总」成一个值**。这样既减少了数据量，又提供了对微小位移的容忍度。

### 池化的历史演进

| 年代 | 方法 | 特点 |
|------|------|------|
| 1980 | Neocognitron 的 C 细胞 | 最早的池化雏形 |
| 1998 | LeNet-5 的平均池化 | 取区域平均值 |
| 2012 | AlexNet 的最大池化 | 取区域最大值，成为主流 |
| 2014 | VGG 的小池化窗 | 2×2 池化 + stride=2 |
| 2014 | GoogLeNet 的全局平均池化 | 整张特征图取平均，替代全连接层 |

今天最常用的是**最大池化（Max Pooling）**——简单、有效、不需要参数。

---
## 2. 生活隐喻：信息的浓缩

### 隐喻一：新闻标题 vs 全文

想象你每天早上看新闻：

**不做池化**（读全文）：
- 每篇新闻从头读到尾，5000 字
- 10 篇新闻 = 50000 字
- 耗时 2 小时，大部分内容其实不重要

**做池化**（看标题）：
- 每篇新闻只看标题，15 字
- 10 篇新闻 = 150 字
- 耗时 2 分钟，抓住了核心信息

池化就像「取标题」——保留最重要的信息（最大值），丢弃细节。你不需要知道新闻中每一句话的具体措辞，只需要知道「发生了什么」。

### 隐喻二：照片压缩

你有一张 4000×3000 的高清照片，想发微信给朋友。原图 12MB，太大发不出去。

**不做池化**：硬发原图 → 传输慢、占空间

**做池化**：把照片缩小到 800×600 → 只有 200KB，但朋友仍然能看清照片里是谁、在哪、在干什么。

池化就是「缩小图片」——保留关键视觉信息，丢弃不重要的像素细节。缩小后的照片虽然模糊了一点，但核心内容一目了然。

### 隐喻三：班级选举

一个班有 40 个学生，要选一个代表去参加学校会议。

**最大池化**的做法：
- 选「嗓门最大」的那个学生（最大值）
- 优点：最突出的特征被保留
- 缺点：可能忽略其他有用的信息

**平均池化**的做法：
- 让所有学生投票，取「平均意见」（平均值）
- 优点：综合了所有人的信息
- 缺点：最突出的特征可能被「平均掉」

在图像中：
- **最大池化**保留最强的特征响应（如最明显的边缘）
- **平均池化**保留区域的平均特征强度（如整体纹理）

---
## 3. 数学直觉：池化到底做了什么？

### 最大池化（Max Pooling）

最大池化的操作非常简单：**把特征图分成若干个小块（如 2×2），每块只保留最大值**。

```
原始特征图 (4×4):        2×2 最大池化后 (2×2):
┌─────────────┐          ┌──────┬──────┐
│ 1  3  2  4  │          │      │      │
│ 5  6  7  8  │    →     │  6   │  8   │
│ 9  2  1  0  │          ├──────┼──────┤
│ 3  4  5  2  │          │  9   │  5   │
└─────────────┘          └──────┴──────┘

左上角 2×2 块 [1,3; 5,6] → max = 6
右上角 2×2 块 [2,4; 7,8] → max = 8
左下角 2×2 块 [9,2; 3,4] → max = 9
右下角 2×2 块 [1,0; 5,2] → max = 5
```

### 平均池化（Average Pooling）

同样的分块方式，但取**平均值**而非最大值：

```
左上角 2×2 块 [1,3; 5,6] → avg = (1+3+5+6)/4 = 3.75
右上角 2×2 块 [2,4; 7,8] → avg = 5.25
左下角 2×2 块 [9,2; 3,4] → avg = 4.5
右下角 2×2 块 [1,0; 5,2] → avg = 2.0
```

### 池化的三个关键参数

```python
nn.MaxPool2d(kernel_size, stride=None, padding=0)
```

| 参数 | 含义 | 常用值 |
|------|------|--------|
| `kernel_size` | 池化窗口大小 | 2（2×2 窗口） |
| `stride` | 窗口滑动步长 | 默认等于 kernel_size（不重叠） |
| `padding` | 边缘补零 | 通常为 0 |

最常用的配置：`MaxPool2d(2, 2)`——2×2 窗口，步长 2。效果是特征图的长宽各减半，面积变为原来的 1/4。

### 池化的输出尺寸

$$H_{out} = \left\lfloor \frac{H + 2p - k}{s} \right\rfloor + 1$$

例如：26×26 的特征图，2×2 池化，stride=2：
- $H_{out} = \lfloor (26 - 2) / 2 \rfloor + 1 = 13$

### 池化的三大好处

1. **降维**：减少特征图尺寸 → 减少后续层的计算量
2. **平移不变性**：特征稍微移动，池化后的最大值/平均值基本不变
3. **防止过拟合**：减少参数和计算 → 降低过拟合风险

### 平移不变性的直观理解

这是池化最精妙的好处。看下面两张特征图——同一张图，只是特征位置偏移了一个像素：

```
特征图 A:              特征图 B (右移1像素):
[0, 0, 0, 0]          [0, 0, 0, 0]
[0, 9, 0, 0]          [0, 0, 9, 0]   ← 强响应右移了
[0, 0, 0, 0]          [0, 0, 0, 0]
[0, 0, 0, 0]          [0, 0, 0, 0]

2×2 最大池化后:
[9, 0]                 [9, 0]         ← 结果完全一样！
[0, 0]                 [0, 0]
```

虽然强响应的精确位置变了，但池化后最大值仍然在左上角——**池化对微小的位置变化不敏感**。

这就是为什么 CNN 能容忍图像中物体的微小位移——池化层在逐层「模糊」精确位置，只保留「大致在哪」。

---
## 4. 代码实现：手写池化

和卷积一样，先手写一个最朴素的池化实现，理解底层逻辑。

In [ ]:
# 设置中文字体
import matplotlib.pyplot as plt
try:
    plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'PingFang SC', 'Helvetica Neue', 'Heiti SC']
    plt.rcParams['axes.unicode_minus'] = False
except:
    pass

import torch

def max_pool2d_naive(feature_map, pool_size=2, stride=2):
    """最朴素的最大池化实现。

    参数:
        feature_map: 输入特征图，形状 [H, W]
        pool_size: 池化窗口大小
        stride: 滑动步长
    返回:
        output: 池化输出
    """
    H, W = feature_map.shape
    out_H = (H - pool_size) // stride + 1  # 输出高度
    out_W = (W - pool_size) // stride + 1  # 输出宽度

    output = torch.zeros(out_H, out_W)

    for i in range(out_H):
        for j in range(out_W):
            # 取出当前池化窗口
            h_start = i * stride
            w_start = j * stride
            window = feature_map[h_start:h_start+pool_size,
                                 w_start:w_start+pool_size]
            # 取窗口中的最大值
            output[i, j] = torch.max(window)

    return output

# 测试
test_map = torch.tensor([
    [1.0, 3.0, 2.0, 4.0],
    [5.0, 6.0, 7.0, 8.0],
    [9.0, 2.0, 1.0, 0.0],
    [3.0, 4.0, 5.0, 2.0],
])

pooled = max_pool2d_naive(test_map, pool_size=2, stride=2)

print("原始特征图 (4×4)：")
print(test_map)
print(f"\n2×2 最大池化后 (2×2)：")
print(pooled)
print(f"\n尺寸变化: {test_map.shape} → {pooled.shape}")
print(f"数据量减少: {test_map.numel()} → {pooled.numel()} (减少 75%)")

### 可视化：池化的效果

用一张真实的纹理图像来展示池化的降维效果。

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np

try:
    plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'PingFang SC', 'Helvetica Neue', 'Heiti SC']
    plt.rcParams['axes.unicode_minus'] = False
except:
    pass

# 创建一张模拟的「纹理」特征图——有明暗交替的条纹
np.random.seed(42)
feature_map = torch.zeros(16, 16)
# 添加一些「特征响应」——模拟卷积后的特征图
for i in range(16):
    for j in range(16):
        # 中心区域有强响应，边缘弱
        dist = ((i - 7.5)**2 + (j - 7.5)**2) ** 0.5
        feature_map[i, j] = max(0, 10 - dist) + np.random.randn() * 0.5

# PyTorch 的池化层
max_pool = nn.MaxPool2d(2, 2)  # 2×2 最大池化
avg_pool = nn.AvgPool2d(2, 2)  # 2×2 平均池化

# 需要 [C, H, W] 格式
fm_4d = feature_map.unsqueeze(0).unsqueeze(0)  # [1, 1, 16, 16]

max_result = max_pool(fm_4d)[0, 0]  # [8, 8]
avg_result = avg_pool(fm_4d)[0, 0]  # [8, 8]

# 可视化
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

im0 = axes[0].imshow(feature_map.numpy(), cmap='viridis')
axes[0].set_title(f'原始特征图\n16×16 = 256 个值')
axes[0].axis('off')
plt.colorbar(im0, ax=axes[0], shrink=0.8)

im1 = axes[1].imshow(max_result.numpy(), cmap='viridis')
axes[1].set_title(f'最大池化后\n8×8 = 64 个值（保留峰值）')
axes[1].axis('off')
plt.colorbar(im1, ax=axes[1], shrink=0.8)

im2 = axes[2].imshow(avg_result.numpy(), cmap='viridis')
axes[2].set_title(f'平均池化后\n8×8 = 64 个值（保留均值）')
axes[2].axis('off')
plt.colorbar(im2, ax=axes[2], shrink=0.8)

plt.suptitle('池化的降维效果：数据量减少 75%', fontsize=14)
plt.tight_layout()
plt.savefig('Week03/images/cnn_day03_pooling_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("观察要点：")
print("  最大池化：保留了最强的响应（亮黄色区域更突出）")
print("  平均池化：整体更平滑，但峰值被「稀释」了")
print("  两种池化都将数据量减少了 75%（256 → 64）")

---
## 5. 验证实验：平移不变性

用一个具体的实验来验证池化的平移不变性——同一个模式出现在不同位置，池化后的输出是否一致？

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

try:
    plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'PingFang SC', 'Helvetica Neue', 'Heiti SC']
    plt.rcParams['axes.unicode_minus'] = False
except:
    pass

# 创建三张特征图：同一个「强响应」出现在不同位置
H, W = 8, 8
fm_center = torch.zeros(H, W)
fm_center[3:5, 3:5] = 10.0  # 强响应在中心

fm_shifted = torch.zeros(H, W)
fm_shifted[3:5, 4:6] = 10.0  # 强响应右移 1 像素

fm_corner = torch.zeros(H, W)
fm_corner[0:2, 0:2] = 10.0  # 强响应在左上角

# 池化
pool = nn.MaxPool2d(2, 2)

def pool_and_show(fm, name):
    fm_4d = fm.unsqueeze(0).unsqueeze(0)
    result = pool(fm_4d)[0, 0]
    return result

r_center = pool_and_show(fm_center, '中心')
r_shifted = pool_and_show(fm_shifted, '右移1像素')
r_corner = pool_and_show(fm_corner, '左上角')

# 可视化
fig, axes = plt.subplots(2, 3, figsize=(14, 8))

for idx, (fm, name, result) in enumerate([
    (fm_center, '强响应在中心', r_center),
    (fm_shifted, '强响应右移 1 像素', r_shifted),
    (fm_corner, '强响应在左上角', r_corner),
]):
    axes[0, idx].imshow(fm.numpy(), cmap='YlOrRd', vmin=0, vmax=10)
    axes[0, idx].set_title(f'原始: {name}')
    axes[0, idx].axis('off')

    axes[1, idx].imshow(result.numpy(), cmap='YlOrRd', vmin=0, vmax=10)
    axes[1, idx].set_title(f'池化后 (4×4)')
    axes[1, idx].axis('off')

plt.suptitle('平移不变性验证：微小位移不影响池化结果', fontsize=14)
plt.tight_layout()
plt.savefig('Week03/images/cnn_day03_original_texture.png', dpi=150, bbox_inches='tight')
plt.show()

# 定量对比
diff_center_shifted = (r_center - r_shifted).abs().sum().item()
diff_center_corner = (r_center - r_corner).abs().sum().item()

print("定量对比（差异越小 = 平移不变性越好）：")
print(f"  中心 vs 右移1像素: 差异 = {diff_center_shifted:.1f}")
print(f"  中心 vs 左上角:     差异 = {diff_center_corner:.1f}")
print(f"\n结论：微小位移（1像素）几乎不影响池化结果")
print(f"      大幅位移（到角落）才会改变池化结果")

---
## 翻译词典

| 生活直觉 | 深度学习术语 |
|----------|-------------|
| 新闻标题（只看重点） | 最大池化（Max Pooling） |
| 全班平均分 | 平均池化（Average Pooling） |
| 缩小照片 | 降采样（Downsampling） |
| 猫挪了一点照样认识 | 平移不变性（Translation Invariance） |
| 2×2 窗口取一个值 | 池化窗口（Pooling Window） |
| 整张图取平均替代全连接 | 全局平均池化（Global Average Pooling） |

---
## 今日总结

| 概念 | 直觉理解 |
|------|----------|
| 最大池化 | 每个小窗口只保留最强的特征响应 |
| 平均池化 | 每个小窗口取平均值，保留整体趋势 |
| 降维 | 特征图长宽减半，数据量减少 75% |
| 平移不变性 | 特征稍微移动，池化结果基本不变 |
| 无参数 | 池化层没有任何可学习的参数 |

**池化的三大作用**：
- **降维**：减少计算量，加快训练
- **平移不变性**：让网络对位置不那么敏感
- **防止过拟合**：减少数据量 = 减少过拟合风险

**明日预告**：卷积 + 池化的组合拳已经就绪。明天我们将搭建完整的 **LeNet-5**——1998 年诞生的第一个商用卷积神经网络，在 MNIST 手写数字上实战训练！